# DRIFTX Gaussian experiment — Kaggle GPU

This notebook checks out **one immutable DRIFTX commit**, verifies Kaggle's existing CUDA/PyTorch environment, downloads the official DA3-GIANT-1.1 checkpoint, and runs sequential Gaussian tests C0–C3. A stage is marked PASS only after actual Gaussian predictions, the DA3 renderer/export, and a reopened non-empty Gaussian PLY all validate.

**Before Run All:** in Kaggle, select a GPU accelerator. Set `INPUT_VIDEO` in the configuration cell, use a matching `test3.mp4`, `test6.mp4`, or `test7.mp4` in an attached Kaggle dataset, or use the upload widget when prompted. Leaving it blank triggers automatic discovery and does not fail just because videos are not committed to GitHub.

**Model attribution and license:** DA3-GIANT-1.1 weights are developed by the ByteDance Seed Team, not DRIFTX, and are published under **CC BY-NC 4.0 (non-commercial use only)**. If the Hugging Face repository restricts downloads, accept its terms on the model card and add an `HF_TOKEN` Kaggle Secret; the notebook reads it without printing the token. Review the [official model card](https://huggingface.co/depth-anything/DA3-GIANT-1.1) and [Depth Anything 3 repository](https://github.com/ByteDance-Seed/Depth-Anything-3) before use. The DRIFTX code checkout and model-weight license are separate; this notebook does not grant commercial rights.

Citation: Lin et al., *Depth Anything 3: Recovering the visual space from any views* (2025), [official project](https://github.com/ByteDance-Seed/Depth-Anything-3), [paper](https://arxiv.org/abs/2511.10647).

Reported peak GPU memory is PyTorch `max_memory_allocated` for this process, not total device use. This experiment does **not** measure georeferencing, registration, reprojection error, DSM coverage, GPS RMSE, accuracy, precision, or recall. Valid depth pixels are not geographic coverage.


In [ ]:
from pathlib import Path

# Editable experiment inputs. Keep the repository commit pinned for reproducibility.
DRIFTX_REPO_URL = "https://github.com/hitakshijoshi20072911/drift-3d.git"
DRIFTX_COMMIT = "c686d1a"
INPUT_VIDEO = ""  # Optional absolute/relative path; otherwise search test3/test6/test7.mp4.
MODEL_REPO_ID = "depth-anything/DA3-GIANT-1.1"
MODEL_DIR = Path("/kaggle/working/models/DA3-GIANT-1.1")
REPO_DIR = Path("/kaggle/working/drift-3d")
OUTPUT_ROOT = Path("/kaggle/working/outputs")
DEVICE = "cuda:0"
ALLOW_CONTINUE_AFTER_FAILURE = False  # OOM always stops; C3 is gated on clean C2 success.
SAMPLE_FPS = 1.0

print("DRIFTX repository:", DRIFTX_REPO_URL)
print("Pinned commit:", DRIFTX_COMMIT)
print("Input video override:", INPUT_VIDEO or "automatic discovery")
print("Model:", MODEL_REPO_ID)
print("Device:", DEVICE)
print("Outputs:", OUTPUT_ROOT)


In [ ]:
import os
import subprocess

def run_git(*args, capture_output=False):
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        check=True,
        text=True,
        capture_output=capture_output,
    )

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", DRIFTX_REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").exists():
    raise RuntimeError(f"Refusing to overwrite a non-Git path: {REPO_DIR}")
else:
    run_git("remote", "set-url", "origin", DRIFTX_REPO_URL)

run_git("fetch", "origin", "--prune")
resolved_commit = run_git("rev-parse", f"{DRIFTX_COMMIT}^{{commit}}", capture_output=True).stdout.strip()
if not resolved_commit.startswith(DRIFTX_COMMIT):
    raise RuntimeError(f"Requested commit {DRIFTX_COMMIT!r} resolved unexpectedly to {resolved_commit}")
run_git("checkout", "--detach", resolved_commit)
checked_out = run_git("rev-parse", "HEAD", capture_output=True).stdout.strip()
if checked_out != resolved_commit:
    raise RuntimeError(f"Pinned checkout mismatch: expected {resolved_commit}, found {checked_out}")

branch_candidates = run_git("branch", "-r", "--contains", resolved_commit, capture_output=True).stdout
source_branches = [line.strip().removeprefix("origin/") for line in branch_candidates.splitlines() if line.strip() and "->" not in line]
PINNED_BRANCH = source_branches[0] if source_branches else "detached commit"
print("Repository:", DRIFTX_REPO_URL)
print("Commit:", checked_out)
print("Branch:", PINNED_BRANCH, "(checkout is detached and pinned)")
print("Git status:")
print(run_git("status", "--short", "--branch", capture_output=True).stdout)
os.chdir(REPO_DIR)


In [ ]:
import platform
import subprocess
import sys

try:
    smi = subprocess.run(["nvidia-smi"], check=False, text=True, capture_output=True)
    print("nvidia-smi:")
    print(smi.stdout if smi.stdout else smi.stderr or "nvidia-smi produced no output")
except FileNotFoundError:
    print("nvidia-smi: not found")

import torch
TORCH_VERSION_BEFORE_INSTALL = torch.__version__
CUDA_VERSION = torch.version.cuda
PYTHON_VERSION = platform.python_version()
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = []
GPU_VRAM_GB = []
print("GPU count:", GPU_COUNT)
print("CUDA runtime:", CUDA_VERSION or "unavailable")
print("PyTorch:", TORCH_VERSION_BEFORE_INSTALL)
print("Python:", PYTHON_VERSION)
for index in range(GPU_COUNT):
    properties = torch.cuda.get_device_properties(index)
    name = properties.name
    vram_gb = properties.total_memory / 1_000_000_000
    GPU_NAMES.append(name)
    GPU_VRAM_GB.append(vram_gb)
    print(f"GPU {index}: {name}; VRAM {vram_gb:.2f} GB")
if GPU_COUNT:
    torch.cuda.set_device(0)  # Use cuda:0. Two T4 devices do not pool/combine their VRAM.
    print("Initial device: cuda:0")
else:
    print("GPU unavailable. GPU-only model and experiment stages will be recorded as not run.")


## Install DRIFTX without replacing Kaggle's PyTorch/CUDA

The install cell filters only `torch`, `torchvision`, and `xformers` out of the repository requirements so Kaggle's preinstalled framework stack is preserved. It reads the pinned `gs` optional dependency directly from this checkout's `pyproject.toml`, builds/installs it with `--no-build-isolation`, and verifies that the PyTorch version did not change.


In [ ]:
import subprocess
import sys
import tomllib

SETUP_ERROR = None
GSPLAT_VERSION = "not installed"
if not torch.cuda.is_available():
    SETUP_ERROR = "No CUDA GPU is available; select a Kaggle GPU accelerator and rerun."
    print(SETUP_ERROR)
else:
    try:
        requirements_path = REPO_DIR / "requirements.txt"
        safe_requirements = Path("/kaggle/working/driftx-kaggle-requirements.txt")
        excluded = {"torch", "torchvision", "xformers"}
        kept = []
        for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            package_name = line.split("[", 1)[0].split("<", 1)[0].split(">", 1)[0].split("=", 1)[0].strip().lower()
            if package_name not in excluded:
                kept.append(line)
        safe_requirements.write_text("\n".join(kept) + "\n", encoding="utf-8")
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(safe_requirements)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True)

        with (REPO_DIR / "pyproject.toml").open("rb") as config_file:
            project_config = tomllib.load(config_file)
        gs_requirements = project_config["project"]["optional-dependencies"]["gs"]
        if not gs_requirements:
            raise RuntimeError("The pinned [gs] extra is empty in this commit.")
        for requirement in gs_requirements:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--no-build-isolation", requirement],
                check=True,
            )

        import torch as torch_after_install
        if torch_after_install.__version__ != TORCH_VERSION_BEFORE_INSTALL:
            raise RuntimeError(
                "PyTorch changed during dependency setup: "
                f"{TORCH_VERSION_BEFORE_INSTALL} -> {torch_after_install.__version__}. "
                "Stop and restore Kaggle's original GPU image before proceeding."
            )
        if not torch_after_install.cuda.is_available():
            raise RuntimeError("CUDA became unavailable during dependency setup.")
        import gsplat
        from importlib.metadata import version as package_version
        GSPLAT_VERSION = getattr(gsplat, "__version__", package_version("gsplat"))
        print("gsplat import: OK")
        print("gsplat version:", GSPLAT_VERSION)
        print("PyTorch preserved:", torch_after_install.__version__)
    except Exception as exc:
        SETUP_ERROR = f"{type(exc).__name__}: {exc}"
        print("Dependency setup failed; stages will not be reported as successful.")
        print(SETUP_ERROR)


In [ ]:
import json
import torch

MODEL_LOAD_ERROR = SETUP_ERROR
model = None
MODEL_CAPABILITIES = None
MODEL_DOWNLOAD_PATH = None
if MODEL_LOAD_ERROR is None:
    try:
        from huggingface_hub import snapshot_download
        hf_token = os.environ.get("HF_TOKEN")
        if not hf_token:
            try:
                from kaggle_secrets import UserSecretsClient
                hf_token = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception:
                hf_token = None
        print("Hugging Face secret configured:", bool(hf_token))
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        MODEL_DOWNLOAD_PATH = snapshot_download(
            repo_id=MODEL_REPO_ID,
            local_dir=str(MODEL_DIR),
            token=hf_token,
        )
        config_file = MODEL_DIR / "config.json"
        if not config_file.is_file():
            raise FileNotFoundError(f"Downloaded checkpoint has no config.json: {config_file}")
        from driftx.models import require_gaussian_capability
        MODEL_CAPABILITIES = require_gaussian_capability(MODEL_DIR).to_dict()
        if not MODEL_CAPABILITIES.get("supports_gaussian"):
            raise RuntimeError("The selected checkpoint does not advertise Gaussian capability.")
        from depth_anything_3.api import DepthAnything3
        model = DepthAnything3.from_pretrained(str(MODEL_DIR))
        model = model.to(device=torch.device(DEVICE))
        model.eval()
        print("Model loaded:", MODEL_REPO_ID)
        print("Model directory:", MODEL_DIR)
        print("Gaussian capability:", json.dumps(MODEL_CAPABILITIES, indent=2))
        print("Prediction.gaussians will be required and checked on every stage inference.")
    except Exception as exc:
        MODEL_LOAD_ERROR = f"{type(exc).__name__}: {exc}"
        print("Model load/capability check failed; no stage can pass.")
        print(MODEL_LOAD_ERROR)
else:
    print("Model download skipped because setup failed:", MODEL_LOAD_ERROR)


In [ ]:
import os
from pathlib import Path

VIDEO_NAMES = {"test3.mp4", "test6.mp4", "test7.mp4"}
SEARCH_ROOTS = [REPO_DIR, Path("/kaggle/working"), Path("/kaggle/input")]

def resolve_explicit_video(value):
    if not value:
        return None
    given = Path(value).expanduser()
    candidates = [given] if given.is_absolute() else [Path.cwd() / given, REPO_DIR / given, Path("/kaggle/input") / given]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"INPUT_VIDEO is set but the file does not exist: {value}")

VIDEO_PATH = resolve_explicit_video(INPUT_VIDEO)
if VIDEO_PATH is None:
    matches = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for path in root.rglob("*"):
            if path.is_file() and path.name.lower() in VIDEO_NAMES:
                matches.append(path.resolve())
    if matches:
        priority = {name: i for i, name in enumerate(("test3.mp4", "test6.mp4", "test7.mp4"))}
        VIDEO_PATH = sorted(matches, key=lambda path: (priority[path.name.lower()], str(path)))[0]

if VIDEO_PATH:
    print("Selected input video:", VIDEO_PATH)
else:
    print("No test3.mp4, test6.mp4, or test7.mp4 found in the repository, Kaggle working, or Kaggle input.")
    print("Upload one in the next cell or set INPUT_VIDEO to an existing path, then rerun the experiment cell.")


### Optional upload cell

If automatic discovery found no input, upload an MP4 here. The experiment cell reads this widget after upload. Alternatively, attach a Kaggle dataset and set `INPUT_VIDEO` to its file path above.


In [ ]:
UPLOAD_WIDGET = None
try:
    import ipywidgets as widgets
    from IPython.display import display
    UPLOAD_WIDGET = widgets.FileUpload(accept=".mp4,.mov,.avi,.mkv,.webm,.m4v", multiple=False)
    display(UPLOAD_WIDGET)
except Exception as exc:
    print("Upload widget unavailable:", exc)
    print("Use Kaggle Add Input to attach a video dataset, or set INPUT_VIDEO to its path.")

def uploaded_video_path():
    if UPLOAD_WIDGET is None:
        return None
    value = UPLOAD_WIDGET.value
    if not value:
        return None
    entries = list(value.values()) if isinstance(value, dict) else list(value)
    if not entries:
        return None
    entry = entries[0]
    name = entry.get("name", "uploaded-video.mp4")
    content = entry.get("content")
    if content is None:
        return None
    suffix = Path(name).suffix.lower() or ".mp4"
    target = Path("/kaggle/working/uploads") / (Path(name).name)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(bytes(content))
    return target.resolve()


## Sequential Gaussian stages

C0: 4 frames at 288 px. C1: 8 frames at 384 px. C2: 16 frames at 384 px. C3: 32 frames at 384 px, and only if C2 passes without OOM. The actual `Prediction.gaussians` object is checked before the repository's Gaussian PLY/video exporter runs. Each output directory contains a `run_report.json` and `metrics.json`, including failures.


In [ ]:
import gc
import json
import math
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import torch

STAGE_PLAN = [("C0", 4, 288), ("C1", 8, 384), ("C2", 16, 384), ("C3", 32, 384)]
STAGE_RESULTS = {}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGE_DIRS = {name: OUTPUT_ROOT / f"cloud_{name.lower()}" for name, _, _ in STAGE_PLAN}
# Start clean so an old artifact can never make a failed rerun appear successful.
for stage_dir in STAGE_DIRS.values():
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)

def is_cuda_oom(exc):
    oom_type = getattr(torch.cuda, "OutOfMemoryError", None)
    return (oom_type is not None and isinstance(exc, oom_type)) or "out of memory" in str(exc).lower()

def artifact_sizes(directory):
    return {str(path.relative_to(directory)): path.stat().st_size for path in directory.rglob("*") if path.is_file() and path.name not in {"run_report.json", "metrics.json"}}

class CachedPredictionModel:
    def __init__(self, prediction):
        self.prediction = prediction
    def inference(self, **kwargs):
        if not kwargs.get("infer_gs"):
            raise RuntimeError("Gaussian exporter did not request infer_gs=True")
        return self.prediction

selected_video = VIDEO_PATH or uploaded_video_path()
if selected_video is not None:
    VIDEO_PATH = Path(selected_video).resolve()
    print("Experiment input:", VIDEO_PATH)

stop_reason = None
for stage_name, frame_limit, resolution in STAGE_PLAN:
    stage_dir = STAGE_DIRS[stage_name]
    if stage_name == "C3":
        c2 = STAGE_RESULTS.get("C2", {})
        if c2.get("status") != "PASS" or c2.get("oom"):
            STAGE_RESULTS[stage_name] = {"status": "SKIPPED", "reason": "C3 requires C2 success without OOM."}
            continue
    if selected_video is None:
        STAGE_RESULTS[stage_name] = {"status": "NOT RUN", "reason": "No video selected or uploaded."}
        continue
    if MODEL_LOAD_ERROR is not None or model is None or not torch.cuda.is_available():
        STAGE_RESULTS[stage_name] = {"status": "FAIL" if stage_name == "C0" else "SKIPPED", "reason": MODEL_LOAD_ERROR or "CUDA/model unavailable.", "oom": False}
        stop_reason = "setup or model load failed"
        if not ALLOW_CONTINUE_AFTER_FAILURE:
            break
        continue
    if stop_reason and not ALLOW_CONTINUE_AFTER_FAILURE:
        STAGE_RESULTS[stage_name] = {"status": "SKIPPED", "reason": stop_reason}
        continue

    started = time.perf_counter()
    prediction = None
    report = {
        "schema_version": 1,
        "stage": stage_name,
        "status": "failed",
        "model": MODEL_REPO_ID,
        "model_capability_verified": bool(MODEL_CAPABILITIES and MODEL_CAPABILITIES.get("supports_gaussian")),
        "device": DEVICE,
        "gpu": GPU_NAMES[0] if GPU_NAMES else "unavailable",
        "gpu_vram_gb": GPU_VRAM_GB[0] if GPU_VRAM_GB else None,
        "cuda": CUDA_VERSION,
        "pytorch": TORCH_VERSION_BEFORE_INSTALL,
        "python": PYTHON_VERSION,
        "gsplat_version": GSPLAT_VERSION,
        "input_video": str(VIDEO_PATH),
        "frames_requested": frame_limit,
        "process_resolution": resolution,
        "sample_fps": SAMPLE_FPS,
        "video_duration_seconds": None,
        "source_fps": None,
        "source_frames": None,
        "sampled_fps": None,
        "frames_extracted": None,
        "frames_processed": None,
        "frame_extraction_time_seconds": None,
        "inference_time_seconds": None,
        "export_time_seconds": None,
        "total_wall_clock_seconds": None,
        "frames_per_second": None,
        "peak_gpu_memory_mb": None,
        "oom": False,
        "error_message": None,
        "gaussian_prediction_nonempty": False,
        "gaussian_renderer_export_validated": False,
        "gaussian_ply_reopened": False,
        "artifacts": {},
        "output_sizes_bytes": {},
    }
    try:
        from plyfile import PlyData
        from driftx.ingest import extract_video_frames
        from driftx.reconstruction.gaussian import run_gaussian_reconstruction
        decode_started = time.perf_counter()
        info, frame_paths = extract_video_frames(VIDEO_PATH, stage_dir, SAMPLE_FPS, frame_limit)
        report["frame_extraction_time_seconds"] = time.perf_counter() - decode_started
        report.update({
            "video_duration_seconds": float(info.duration_seconds),
            "source_fps": float(info.source_fps),
            "source_frames": int(info.source_frames),
            "sampled_fps": float(info.sampled_fps),
            "frames_extracted": len(frame_paths),
        })

        torch.cuda.synchronize(0)
        torch.cuda.reset_peak_memory_stats(0)
        inference_started = time.perf_counter()
        with torch.inference_mode():
            prediction = model.inference(
                image=frame_paths,
                infer_gs=True,
                export_dir=None,
                export_format="mini_npz",
                process_res=resolution,
            )
        torch.cuda.synchronize(0)
        report["inference_time_seconds"] = time.perf_counter() - inference_started
        # Validate Prediction.gaussians from this actual inference call, not merely gsplat import.
        gaussians = prediction.gaussians
        means = getattr(gaussians, "means", None) if gaussians is not None else None
        if means is None or not hasattr(means, "numel") or means.numel() == 0:
            raise RuntimeError("Actual inference returned no non-empty Prediction.gaussians.means.")
        if not torch.isfinite(means).all().item():
            raise RuntimeError("Prediction.gaussians.means contains non-finite values.")
        gaussian_count = int(means.numel() // 3)
        if gaussian_count <= 0:
            raise RuntimeError("Prediction.gaussians contains zero Gaussian points.")
        report["gaussian_prediction_nonempty"] = True
        report["gaussian_count"] = gaussian_count
        report["frames_processed"] = len(frame_paths)

        export_started = time.perf_counter()
        _, gaussian_artifacts = run_gaussian_reconstruction(
            CachedPredictionModel(prediction),
            frame_paths,
            stage_dir,
            process_res=resolution,
        )
        report["export_time_seconds"] = time.perf_counter() - export_started
        gaussian_ply = Path(gaussian_artifacts["gaussian_ply"])
        gaussian_video = Path(gaussian_artifacts["gaussian_preview"])
        for required in (gaussian_ply, gaussian_video):
            if not required.is_file() or required.stat().st_size == 0:
                raise RuntimeError(f"Requested Gaussian export is missing or empty: {required}")

        reopened = PlyData.read(str(gaussian_ply))
        vertex = reopened["vertex"]
        if len(vertex.data) <= 0 or not {"x", "y", "z"}.issubset(vertex.data.dtype.names or ()):
            raise RuntimeError("Gaussian PLY reopened but has no non-empty XYZ vertex data.")
        report["gaussian_renderer_export_validated"] = True
        report["gaussian_ply_reopened"] = True

        arrays = {"depth": np.asarray(prediction.depth)}
        if getattr(prediction, "conf", None) is not None:
            arrays["confidence"] = np.asarray(prediction.conf)
        depth_path = stage_dir / "depth_confidence.npz"
        np.savez_compressed(depth_path, **arrays)
        report["artifacts"] = {
            "gaussian_ply": str(gaussian_ply),
            "gaussian_preview": str(gaussian_video),
            "depth_confidence": str(depth_path),
        }
        report["status"] = "completed"
        report["peak_gpu_memory_mb"] = round(torch.cuda.max_memory_allocated(0) / (1024 ** 2), 2)
        report["peak_gpu_memory_metric"] = "PyTorch max_memory_allocated for this process; excludes other processes/non-PyTorch memory."
        report["frames_per_second"] = report["frames_processed"] / max(time.perf_counter() - started, 1e-9)
        depth_values = arrays["depth"]
        valid_depth = np.isfinite(depth_values) & (depth_values > 0)
        report["valid_depth_pixels"] = int(valid_depth.sum())
        report["valid_depth_percentage"] = float(100.0 * valid_depth.mean())
        report["valid_depth_definition"] = "Finite positive depth values; not geographic coverage."
        confidence_values = arrays.get("confidence")
        if confidence_values is not None:
            finite_confidence = confidence_values[np.isfinite(confidence_values)]
            if finite_confidence.size:
                report["mean_confidence"] = float(finite_confidence.mean())
                report["median_confidence"] = float(np.median(finite_confidence))
    except Exception as exc:
        report["status"] = "failed"
        report["oom"] = is_cuda_oom(exc)
        report["error_message"] = f"{type(exc).__name__}: {exc}"
        report["error_traceback"] = traceback.format_exc()
        if report["oom"]:
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
    finally:
        report["total_wall_clock_seconds"] = time.perf_counter() - started
        report["output_sizes_bytes"] = artifact_sizes(stage_dir)
        report_path = stage_dir / "run_report.json"
        metrics_path = stage_dir / "metrics.json"
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        metrics = {key: value for key, value in report.items() if key not in {"error_traceback"}}
        metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    stage_result = {
        "status": "PASS" if report["status"] == "completed" and report["gaussian_prediction_nonempty"] and report["gaussian_renderer_export_validated"] and report["gaussian_ply_reopened"] else "FAIL",
        "oom": bool(report["oom"]),
        "run_report": str(report_path),
        "metrics": str(metrics_path),
        "artifacts": report.get("artifacts", {}),
        "error": report.get("error_message"),
    }
    STAGE_RESULTS[stage_name] = stage_result
    print(f"{stage_name}: {stage_result['status']} | frames={report.get('frames_processed', 'not measured')} | res={resolution} | OOM={report['oom']} | runtime={report.get('total_wall_clock_seconds', 'not measured')} s | error={report.get('error_message') or '-'}")
    if stage_result["status"] != "PASS":
        stop_reason = f"{stage_name} failed; stopping by default."
        if report["oom"] or not ALLOW_CONTINUE_AFTER_FAILURE:
            break
    if prediction is not None:
        del prediction
    gc.collect()
    torch.cuda.empty_cache()

for stage_name, _, _ in STAGE_PLAN:
    if stage_name not in STAGE_RESULTS:
        STAGE_RESULTS[stage_name] = {"status": "SKIPPED", "reason": stop_reason or "Not reached."}
    result = STAGE_RESULTS[stage_name]
    if not result.get("run_report"):
        stage_dir = STAGE_DIRS[stage_name]
        status = result.get("status", "SKIPPED")
        report_status = {"PASS": "completed", "FAIL": "failed", "NOT RUN": "not_run", "SKIPPED": "skipped"}.get(status, "skipped")
        report = {
            "schema_version": 1, "stage": stage_name, "status": report_status,
            "model": MODEL_REPO_ID, "device": DEVICE,
            "gpu": GPU_NAMES[0] if GPU_NAMES else "unavailable",
            "gsplat_version": GSPLAT_VERSION, "input_video": str(VIDEO_PATH) if VIDEO_PATH else None,
            "error_message": result.get("reason") or result.get("error"),
            "oom": bool(result.get("oom", False)), "artifacts": {}, "output_sizes_bytes": {},
        }
        report_path = stage_dir / "run_report.json"
        metrics_path = stage_dir / "metrics.json"
        report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        metrics_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        result["run_report"] = str(report_path)
        result["metrics"] = str(metrics_path)


In [ ]:
import json
import zipfile
from pathlib import Path

vram_text = ", ".join(f"{name} ({size:.2f} GB)" for name, size in zip(GPU_NAMES, GPU_VRAM_GB)) or "unavailable"
successful_artifacts = []
failed_artifacts = []
for stage_name, result in STAGE_RESULTS.items():
    if result.get("status") == "PASS":
        successful_artifacts.extend(result.get("artifacts", {}).values())
    elif result.get("status") == "FAIL":
        failed_artifacts.extend(path for path in (result.get("run_report"), result.get("metrics")) if path)

summary = {
    "schema_version": 1,
    "repository": DRIFTX_REPO_URL,
    "commit": run_git("rev-parse", "HEAD", capture_output=True).stdout.strip(),
    "branch": PINNED_BRANCH,
    "git_status": run_git("status", "--short", "--branch", capture_output=True).stdout.strip(),
    "gpu": GPU_NAMES[0] if GPU_NAMES else "unavailable",
    "gpu_count": GPU_COUNT,
    "gpu_vram": vram_text,
    "cuda": CUDA_VERSION,
    "pytorch": TORCH_VERSION_BEFORE_INSTALL,
    "python": PYTHON_VERSION,
    "model": MODEL_REPO_ID,
    "model_load_error": MODEL_LOAD_ERROR,
    "gsplat": GSPLAT_VERSION,
    "input_video": str(VIDEO_PATH) if VIDEO_PATH else None,
    "stages": STAGE_RESULTS,
    "successful_artifacts": successful_artifacts,
    "failed_artifacts": failed_artifacts,
}
summary_path = OUTPUT_ROOT / "driftx_gaussian_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")

archive_path = Path("/kaggle/working/driftx_gaussian_artifacts.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for stage_name, result in STAGE_RESULTS.items():
        if result.get("status") != "PASS":
            continue
        stage_dir = STAGE_DIRS[stage_name]
        for path in sorted(stage_dir.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(Path("/kaggle/working")))

print("DRIFTX GAUSSIAN EXPERIMENT SUMMARY")
print("Repository:", summary["repository"])
print("Commit:", summary["commit"])
print("GPU:", summary["gpu"])
print("VRAM:", summary["gpu_vram"])
print("CUDA:", summary["cuda"])
print("PyTorch:", summary["pytorch"])
print("Python:", summary["python"])
print("Model:", summary["model"])
print("gsplat:", summary["gsplat"])
for stage_name, _, _ in STAGE_PLAN:
    result = STAGE_RESULTS[stage_name]
    print(f"{stage_name}: {result.get('status', 'NOT RUN')}" + (f" — {result.get('reason') or result.get('error')}" if result.get("reason") or result.get("error") else ""))
print("Successful artifacts:")
for path in successful_artifacts:
    print(" -", path)
print("Failed artifacts:")
for path in failed_artifacts:
    print(" -", path)
print("Summary JSON:", summary_path.resolve())
print("Artifact archive:", archive_path.resolve())
print("Summary status file:", summary_path.resolve())
